← [Previous: Chapter 5 — ibex RISC-V Core](https://colab.research.google.com/github/najaeda/naja/blob/main/tutorials/notebooks/05_ibex_riscv_core.ipynb) · 🔗 [Back to Table of Contents](https://github.com/najaeda/naja/tree/main/tutorials)

# Chapter 6: Gate-Level Analysis — Fanout and Net Statistics

**Fanout** is the number of gate inputs driven by a single net. High-fanout nets are a root cause of timing closure failures in ASIC and FPGA designs: one driver charging many inputs creates large capacitive loads that slow signal propagation.

Identifying and understanding high-fanout nets is a task EDA engineers typically perform with proprietary sign-off tools. This chapter shows how to do the same analysis in 50 lines of Python using `najaeda`.

We will use the same **TinyRocket** synthesised design from Chapter 2. By the end you will have:
1. Computed the fanout of every net in the design using the **Equipotential** API
2. Found the top high-fanout nets
3. Traced a high-fanout net back to its driver
4. Exported results to a pandas DataFrame and plotted the fanout distribution

In [ ]:
# Colab / local setup — skipped in CI
import os
from pathlib import Path

!wget https://github.com/najaeda/naja/archive/refs/heads/main.zip
!unzip main.zip
os.environ['NAJAEDA_TUTORIALS_BENCHMARKS'] = 'naja-main/tutorials/benchmarks'
!pip install najaeda==0.7.6 pandas matplotlib

## Loading the Design

We load the TinyRocket gate-level netlist with its Liberty standard-cell library, exactly as in Chapter 2.

In [ ]:
import os
from pathlib import Path
from najaeda import netlist

_benchmarks = os.environ.get(
    'NAJAEDA_TUTORIALS_BENCHMARKS',
    str(Path().resolve().parent / 'benchmarks')
)
liberty_files = [
    f'{_benchmarks}/liberty/tutorial_cells.lib',
    f'{_benchmarks}/liberty/fakeram45_64x32.lib',
]
netlist.load_liberty(liberty_files)
top = netlist.load_verilog(f'{_benchmarks}/verilog/tinyrocket/tinyrocket.v')
print(f"Loaded: {top.get_name()}")
print(f"Direct children: {top.count_child_instances()}")

---

## What is an Equipotential?

A **net** in najaeda represents a wire *within a single module*. When a signal crosses a module boundary (passes through a port), it appears as separate net objects in parent and child modules.

An **equipotential** is the flat, cross-hierarchy view of the same signal: all the terminal connections that are electrically equivalent, regardless of where they sit in the hierarchy. It is the right abstraction for fanout analysis.

```
top module
│  net "clk" ─── port clk_i of sub_inst
│
└─ sub_inst (model: cpu_core)
       net "clk_i" ─── input of reg_A, input of reg_B, ...
```

`clk` and `clk_i` are different net objects, but belong to the *same* equipotential. Fanout of `clk` = all inputs reachable through the equipotential.

## Computing Fanout for Every Net

We iterate every **bit-level net** (`get_bit_nets()` expands bus nets into individual bits so each slice can be independently analysed) and use `get_equipotential()` to find all connected instance terminals across the full hierarchy. The number of those terminals minus one (the driver) gives the fanout.

In [ ]:
from najaeda import instance_visitor

fanout_records = []

for net in top.get_bit_nets():
    net_terms = list(net.get_terms())
    if not net_terms:
        continue

    # Any terminal on this net gives access to the full equipotential
    equi = net_terms[0].get_equipotential()
    inst_terms = list(equi.get_inst_terms())

    # inst_terms includes the driver; fanout = loads only
    fanout = len(inst_terms) - 1
    if fanout > 0:
        fanout_records.append({
            'net': net.get_name() or '<anonymous>',
            'fanout': fanout,
        })

fanout_records.sort(key=lambda r: -r['fanout'])
print(f"Nets with fanout > 0 : {len(fanout_records)}")
print(f"Max fanout           : {fanout_records[0]['fanout']}  ({fanout_records[0]['net']})")
print(f"Mean fanout          : {sum(r['fanout'] for r in fanout_records) / len(fanout_records):.1f}")

## Top 20 High-Fanout Nets

In [ ]:
print(f"{'Fanout':>7}  Net name")
print("-" * 60)
for rec in fanout_records[:20]:
    print(f"  {rec['fanout']:>5}  {rec['net']}")

## Tracing the Driver of the Highest-Fanout Net

For a high-fanout net we want to know: *what is driving it?* We use the equipotential to find the terminal whose direction is **Output** — that is the driver cell and pin.

In [ ]:
# Take the highest-fanout net
top_net_name = fanout_records[0]['net']
top_net = top.get_net(top_net_name)

# Get the equipotential from one of its terminals
equi = list(top_net.get_terms())[0].get_equipotential()

# Separate drivers (output terminals) from loads (input terminals)
drivers = []
loads   = []
for iterm in equi.get_inst_terms():
    direction = str(iterm.get_direction())
    if 'Output' in direction:
        drivers.append(iterm)
    else:
        loads.append(iterm)

print(f"Net '{top_net_name}'  fanout = {len(loads)}")
print(f"\nDriver(s):")
for d in drivers:
    print(f"  {d.get_instance().get_name()} / {d.get_instance().get_model_name()}  pin: {d.get_name()}")
print(f"\nFirst 10 loads:")
for l in loads[:10]:
    print(f"  {l.get_instance().get_name()} / {l.get_instance().get_model_name()}  pin: {l.get_name()}")

## Exporting to Pandas

All fanout records can be loaded into a DataFrame for further analysis — filtering, grouping, or joining with other EDA data.

In [ ]:
import pandas as pd

df = pd.DataFrame(fanout_records)
print(df.describe())
print()
print("Fanout distribution (bucket counts):")
print(df['fanout'].value_counts().sort_index().head(20))

## Fanout Distribution Histogram

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['fanout'], bins=40, log=True, color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Fanout', fontsize=12)
ax.set_ylabel('Number of nets (log scale)', fontsize=12)
ax.set_title(f'Fanout distribution — {top.get_name()}', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.axvline(20, color='firebrick', linestyle='--', linewidth=1.2, label='fanout = 20')
ax.legend()
plt.tight_layout()
plt.savefig('fanout_distribution.png', dpi=150)
plt.show()
print("Saved: fanout_distribution.png")

## Saving Results to CSV

Export the full results for use in spreadsheets, reports, or downstream scripts.

In [ ]:
df.to_csv('fanout_report.csv', index=False)
print(f"Written: fanout_report.csv  ({len(df)} rows)")
print(df[df['fanout'] >= 20].to_string(index=False))

---

✅ This concludes **Chapter 6** of the **najaeda** tutorial.

With fewer than 50 lines of Python you have:
- Computed the fanout of every net in a real synthesised design using the Equipotential API
- Identified the top high-fanout nets and traced their drivers
- Exported results to pandas and produced a visualisation

**What to do with this data in practice:**
- Nets with fanout > 20 are candidates for buffer insertion
- Clock nets with very high fanout are expected — check that they are driven by clock buffers
- Nets with unexpectedly high fanout may indicate a missing clock-tree synthesis step

📚 [najaeda API docs](https://najaeda.readthedocs.io/en/latest/) — see the `Equipotential` and `instance_visitor` reference pages for more traversal options.